# Kaggle QA Chroma Builder

This notebook builds the full QA retrieval index on Kaggle GPU.

Flow:

```text
raw JSON dataset -> clean QA chunks -> E5 embeddings -> Chroma DB -> zip output
```

After it finishes, download `chroma_db_qa.zip` and extract it to `D:/Ds107/chroma_db_qa` on your local machine.

In [ ]:
!pip install -q langchain-core langchain-chroma langchain-huggingface sentence-transformers chromadb

In [ ]:
# =============================================================================
# 1. Imports and configuration
# =============================================================================

from __future__ import annotations

import json
import os
import re
import shutil
import unicodedata
import zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


# Set DATASET_PATH manually if auto-discovery cannot find your file.
DATASET_PATH = None

PERSIST_DIR = Path('/kaggle/working/chroma_db_qa')
ZIP_PATH = Path('/kaggle/working/chroma_db_qa.zip')

COLLECTION_NAME = 'qa_chunks'
EMBED_MODEL = 'intfloat/multilingual-e5-base'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Use None for full dataset. Use a small number like 500 for smoke testing.
SAMPLE_SIZE = None

# Chroma add_documents batch size. Reduce this if Kaggle runs out of memory.
INDEX_BATCH_SIZE = 512

# SentenceTransformer encode batch size. Reduce this if GPU memory is tight.
EMBED_BATCH_SIZE = 64

# In Kaggle /working, it is usually safe to rebuild from scratch.
CLEAR_EXISTING_INDEX = True

print('Device:', DEVICE)
print('Torch CUDA available:', torch.cuda.is_available())

In [ ]:
# =============================================================================
# 2. Secrets, paths, and dataset loading
# =============================================================================

def load_hf_token_from_kaggle_secret() -> None:
    """Load HF_TOKEN from Kaggle Secrets when available."""

    if os.environ.get('HF_TOKEN'):
        return

    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return

    secret_client = UserSecretsClient()
    for secret_name in ('HF_TOKEN', 'HUGGINGFACEHUB_API_TOKEN'):
        try:
            token = secret_client.get_secret(secret_name)
        except Exception:
            token = None

        if token:
            os.environ['HF_TOKEN'] = token
            return


def find_dataset_path() -> Path:
    """Find vietnamese_vqa_dataset.json inside /kaggle/input."""

    if DATASET_PATH:
        return Path(DATASET_PATH)

    candidates = list(Path('/kaggle/input').rglob('vietnamese_vqa_dataset.json'))
    if not candidates:
        raise FileNotFoundError(
            'Could not find vietnamese_vqa_dataset.json in /kaggle/input. '
            'Set DATASET_PATH manually in the config cell.'
        )

    return candidates[0]


def load_dataset(dataset_path: str | Path) -> list[dict[str, Any]]:
    """Load the raw JSON dataset."""

    with Path(dataset_path).open('r', encoding='utf-8') as dataset_file:
        return json.load(dataset_file)


load_hf_token_from_kaggle_secret()
os.environ.setdefault('HF_HOME', '/kaggle/working/hf_cache')
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')

dataset_path = find_dataset_path()
print('Dataset path:', dataset_path)
print('HF token loaded:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# =============================================================================
# 3. Clean QA chunking helpers
# =============================================================================

GENERIC_IDENTIFICATION_PATTERNS = (
    'day la gi',
    'day la mon gi',
    'day la loai gi',
    'hinh anh nay la gi',
    'vat the nay la gi',
    'mon an nay la gi',
    'trang phuc nay la gi',
    'nhac cu nay la gi',
)

TOPIC_PREFIXES_TO_REMOVE = (
    'day la',
    'do la',
    'co ve la',
    'hinh anh mo ta',
    'hinh anh the hien',
    'hinh anh nay mo ta',
    'hinh anh nay the hien',
)

MAX_TOPIC_WORDS = 12


@dataclass(frozen=True)
class ResolvedTopic:
    text: str
    source: str


@dataclass(frozen=True)
class QaChunk:
    page_content: str
    metadata: dict[str, Any]


def safe_text(value: Any) -> str:
    """Convert strings, lists, and empty values into a display string."""

    if value is None:
        return ''

    if isinstance(value, list):
        return ', '.join(str(item).strip() for item in value if item)

    return str(value).strip()


def normalize_text(value: Any) -> str:
    """Normalize text for matching and deduplication only."""

    text = safe_text(value).lower()
    if not text:
        return ''

    text = unicodedata.normalize('NFC', text)
    text = ''.join(
        char
        for char in unicodedata.normalize('NFD', text)
        if unicodedata.category(char) != 'Mn'
    )
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def remove_topic_wrapper(topic_candidate: Any) -> str:
    """Remove generic VQA wrappers while preserving Vietnamese text."""

    display_topic = safe_text(topic_candidate).strip(' .,:;!?')
    normalized_topic = normalize_text(display_topic)

    for normalized_prefix in TOPIC_PREFIXES_TO_REMOVE:
        if not normalized_topic.startswith(normalized_prefix + ' '):
            continue

        prefix_word_count = len(normalized_prefix.split())
        display_words = display_topic.split()
        return ' '.join(display_words[prefix_word_count:]).strip(' .,:;!?')

    return display_topic


def is_identification_question(question_record: dict[str, Any]) -> bool:
    """Return True when the answer likely names the actual topic."""

    question_type = normalize_text(question_record.get('question_type'))
    question_text = normalize_text(question_record.get('question'))

    if 'identification' in question_type:
        return True

    return any(pattern in question_text for pattern in GENERIC_IDENTIFICATION_PATTERNS)


def is_usable_topic(topic_candidate: Any) -> bool:
    """Reject empty or overly long topic candidates."""

    normalized_topic = normalize_text(topic_candidate)
    if not normalized_topic:
        return False

    return len(normalized_topic.split()) <= MAX_TOPIC_WORDS


def resolve_topic_from_identification_answer(sample: dict[str, Any]) -> ResolvedTopic | None:
    """Prefer identification answers over noisy keyword labels."""

    for question_record in sample.get('questions', []) or []:
        if not is_identification_question(question_record):
            continue

        topic_text = remove_topic_wrapper(question_record.get('answer'))
        if is_usable_topic(topic_text):
            return ResolvedTopic(text=topic_text, source='identification_answer')

    return None


def resolve_topic_from_primary_object(sample: dict[str, Any]) -> ResolvedTopic | None:
    """Use primary cultural object when QA answer is not enough."""

    cultural_context = sample.get('cultural_context', {}) or {}
    primary_objects = cultural_context.get('primary_cultural_objects', []) or []

    for object_name in primary_objects:
        topic_text = remove_topic_wrapper(object_name)
        if is_usable_topic(topic_text):
            return ResolvedTopic(text=topic_text, source='primary_cultural_object')

    return None


def resolve_canonical_topic(sample: dict[str, Any]) -> ResolvedTopic:
    """Resolve topic using QA answer -> primary object -> keyword fallback."""

    topic_from_answer = resolve_topic_from_identification_answer(sample)
    if topic_from_answer:
        return topic_from_answer

    topic_from_object = resolve_topic_from_primary_object(sample)
    if topic_from_object:
        return topic_from_object

    return ResolvedTopic(
        text=remove_topic_wrapper(sample.get('keyword')),
        source='keyword_fallback',
    )

In [ ]:
# =============================================================================
# 4. Build QA chunks
# =============================================================================

def build_topic_aliases(sample: dict[str, Any], topic: str) -> list[str]:
    """Build alternate names for recall without controlling the topic."""

    cultural_context = sample.get('cultural_context', {}) or {}
    primary_objects = cultural_context.get('primary_cultural_objects', []) or []

    raw_aliases = [
        topic,
        sample.get('keyword'),
        sample.get('category'),
        cultural_context.get('cultural_category'),
        *primary_objects,
    ]

    aliases = []
    seen_normalized_aliases = set()

    for raw_alias in raw_aliases:
        alias = safe_text(raw_alias)
        normalized_alias = normalize_text(alias)

        if not alias or not normalized_alias:
            continue

        if normalized_alias in seen_normalized_aliases:
            continue

        aliases.append(alias)
        seen_normalized_aliases.add(normalized_alias)

    return aliases


def build_page_content(
    sample: dict[str, Any],
    question_record: dict[str, Any],
    resolved_topic: ResolvedTopic,
    topic_aliases: list[str],
) -> str:
    """Create the text that will be embedded into Chroma."""

    cultural_context = sample.get('cultural_context', {}) or {}
    additional_context = question_record.get('additional_context', {}) or {}

    sections = [
        'Chunk Type: qa_chunk',
        f'Canonical Topic: {resolved_topic.text}',
        f'Topic Source: {resolved_topic.source}',
        f'Aliases: {", ".join(topic_aliases)}',
        f'Category: {safe_text(sample.get("category"))}',
        f'Question Type: {safe_text(question_record.get("question_type"))}',
        f'Difficulty: {safe_text(question_record.get("difficulty"))}',
        f'Cognitive Level: {safe_text(question_record.get("cognitive_level"))}',
        '',
        'Question:',
        safe_text(question_record.get('question')),
        '',
        'Answer:',
        safe_text(question_record.get('answer')),
        '',
        'Detailed Explanation:',
        safe_text(question_record.get('detailed_explanation')),
        '',
        'Cultural Significance:',
        safe_text(question_record.get('cultural_significance')),
        '',
        'Historical Context:',
        safe_text(cultural_context.get('historical_context')),
        '',
        'Modern Relevance:',
        safe_text(cultural_context.get('modern_relevance')),
        '',
        'Origin:',
        safe_text(additional_context.get('origin')),
        '',
        'Usage:',
        safe_text(additional_context.get('usage')),
        '',
        'Symbolism:',
        safe_text(additional_context.get('symbolism')),
        '',
        'Regional Variations:',
        safe_text(additional_context.get('regional_variations')),
    ]

    return '\n'.join(sections).strip()


def build_metadata(
    sample: dict[str, Any],
    question_record: dict[str, Any],
    resolved_topic: ResolvedTopic,
    topic_aliases: list[str],
) -> dict[str, Any]:
    """Create metadata for filtering, debugging, and source tracing."""

    question_text = safe_text(question_record.get('question'))
    category = safe_text(sample.get('category'))
    image_id = safe_text(sample.get('image_id'))
    question_id = safe_text(question_record.get('question_id'))

    return {
        'doc_id': f'{category}:{normalize_text(resolved_topic.text)}:{image_id}:{question_id}',
        'chunk_type': 'qa_chunk',
        'category': category,
        'topic': resolved_topic.text,
        'normalized_topic': normalize_text(resolved_topic.text),
        'topic_source': resolved_topic.source,
        'aliases': ' | '.join(topic_aliases),
        'keyword': safe_text(sample.get('keyword')),
        'normalized_keyword': normalize_text(sample.get('keyword')),
        'question': question_text,
        'normalized_question': normalize_text(question_text),
        'question_type': safe_text(question_record.get('question_type')),
        'difficulty': safe_text(question_record.get('difficulty')),
        'cognitive_level': safe_text(question_record.get('cognitive_level')),
        'image_id': image_id,
        'image_path': safe_text(sample.get('image_path')),
    }


def build_qa_chunk(sample: dict[str, Any], question_record: dict[str, Any]) -> QaChunk:
    """Build one clean QA chunk from one sample and one question."""

    resolved_topic = resolve_canonical_topic(sample)
    topic_aliases = build_topic_aliases(sample, resolved_topic.text)

    return QaChunk(
        page_content=build_page_content(sample, question_record, resolved_topic, topic_aliases),
        metadata=build_metadata(sample, question_record, resolved_topic, topic_aliases),
    )


def build_dedup_key(
    sample: dict[str, Any],
    question_record: dict[str, Any],
    resolved_topic: ResolvedTopic,
) -> tuple[str, str, str, str]:
    """Build dedup key that ignores image_id."""

    return (
        safe_text(sample.get('category')),
        normalize_text(resolved_topic.text),
        safe_text(question_record.get('question_type')),
        normalize_text(question_record.get('question')),
    )


def build_qa_chunks(dataset_records: list[dict[str, Any]]) -> list[QaChunk]:
    """Build deduplicated QA chunks from raw dataset records."""

    qa_chunks = []
    seen_dedup_keys = set()

    for sample in dataset_records:
        resolved_topic = resolve_canonical_topic(sample)

        for question_record in sample.get('questions', []) or []:
            if not normalize_text(question_record.get('question')):
                continue

            dedup_key = build_dedup_key(sample, question_record, resolved_topic)
            if dedup_key in seen_dedup_keys:
                continue

            qa_chunks.append(build_qa_chunk(sample, question_record))
            seen_dedup_keys.add(dedup_key)

    return qa_chunks


def summarize_chunks(qa_chunks: list[QaChunk]) -> None:
    """Print compact chunk stats before indexing."""

    print('Total chunks:', len(qa_chunks))
    print('Chunk types:', dict(Counter(chunk.metadata['chunk_type'] for chunk in qa_chunks)))
    print('Topic sources:', dict(Counter(chunk.metadata['topic_source'] for chunk in qa_chunks)))
    print('Categories:', Counter(chunk.metadata['category'] for chunk in qa_chunks).most_common())


def preview_chunks(qa_chunks: list[QaChunk], limit: int = 3) -> None:
    """Show a few chunks before embedding."""

    for index, chunk in enumerate(qa_chunks[:limit], start=1):
        metadata = chunk.metadata
        print('=' * 80)
        print('CHUNK', index)
        print('topic:', metadata['topic'])
        print('topic_source:', metadata['topic_source'])
        print('keyword:', metadata['keyword'])
        print('question:', metadata['question'])
        print(chunk.page_content[:800])


def to_langchain_documents(qa_chunks: list[QaChunk]) -> list[Document]:
    """Convert QA chunks to LangChain Documents for Chroma."""

    return [
        Document(page_content=chunk.page_content, metadata=chunk.metadata)
        for chunk in qa_chunks
    ]

In [ ]:
# =============================================================================
# 5. Load dataset and create documents
# =============================================================================

dataset_records = load_dataset(dataset_path)
print('Raw records:', len(dataset_records))

if SAMPLE_SIZE is not None:
    dataset_records = dataset_records[:SAMPLE_SIZE]
    print('Using sample records:', len(dataset_records))

qa_chunks = build_qa_chunks(dataset_records)
summarize_chunks(qa_chunks)
preview_chunks(qa_chunks, limit=3)

documents = to_langchain_documents(qa_chunks)
print('Documents ready:', len(documents))

In [ ]:
# =============================================================================
# 6. Build Chroma index with E5 embeddings
# =============================================================================

class E5Embeddings:
    """Apply E5 query/passsage prefixes around HuggingFace embeddings."""

    def __init__(self, model_name: str, device: str) -> None:
        self.embedding_model = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': device},
            encode_kwargs={
                'normalize_embeddings': True,
                'batch_size': EMBED_BATCH_SIZE,
            },
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        passages = [f'passage: {text}' for text in texts]
        return self.embedding_model.embed_documents(passages)

    def embed_query(self, text: str) -> list[float]:
        return self.embedding_model.embed_query(f'query: {text}')


def batch_documents(items: list[Document], batch_size: int):
    """Yield fixed-size document batches for Chroma indexing."""

    for start_index in range(0, len(items), batch_size):
        yield start_index, items[start_index:start_index + batch_size]


if CLEAR_EXISTING_INDEX and PERSIST_DIR.exists():
    shutil.rmtree(PERSIST_DIR)

PERSIST_DIR.mkdir(parents=True, exist_ok=True)

embeddings = E5Embeddings(model_name=EMBED_MODEL, device=DEVICE)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)

for start_index, document_batch in batch_documents(documents, INDEX_BATCH_SIZE):
    vectorstore.add_documents(document_batch)
    indexed_count = min(start_index + len(document_batch), len(documents))
    print(f'Indexed {indexed_count}/{len(documents)} documents')

print('Chroma index saved to:', PERSIST_DIR)

In [ ]:
# =============================================================================
# 7. Preview retrieval quality
# =============================================================================

def preview_search(query: str, top_k: int = 5) -> None:
    """Show retrieval results and metadata for one query."""

    results = vectorstore.similarity_search_with_score(query, k=top_k)

    print('=' * 80)
    print('QUERY:', query)

    for rank, (document, score) in enumerate(results, start=1):
        metadata = document.metadata

        print('=' * 80)
        print(f'RANK {rank} | SCORE {score:.6f}')
        print('topic:', metadata.get('topic'))
        print('topic_source:', metadata.get('topic_source'))
        print('keyword:', metadata.get('keyword'))
        print('category:', metadata.get('category'))
        print('question_type:', metadata.get('question_type'))
        print('question:', metadata.get('question'))
        print(document.page_content[:800])


preview_queries = [
    'Ý nghĩa văn hóa của thảm cói là gì?',
    'Xe máy là gì?',
    'So sánh bánh chưng và bánh tét',
]

for query in preview_queries:
    preview_search(query, top_k=5)

In [ ]:
# =============================================================================
# 8. Zip Chroma DB for local download
# =============================================================================

def zip_directory(source_dir: Path, output_zip: Path) -> None:
    """Zip the Chroma directory while preserving relative paths."""

    if output_zip.exists():
        output_zip.unlink()

    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
        for file_path in source_dir.rglob('*'):
            if file_path.is_file():
                zip_file.write(file_path, file_path.relative_to(source_dir.parent))


zip_directory(PERSIST_DIR, ZIP_PATH)
print('Created:', ZIP_PATH)
print('Download this file and extract it to D:/Ds107/chroma_db_qa')